In [ ]:
import importlib.util
import subprocess
import sys

PIP_PACKAGES = [
    "numpy>=1.23.0",
    "pandas>=2.0.0",
    "pyarrow>=10.0.0",
    "pulp>=2.8.0",
    "plotly>=5.18.0",
    "kaleido==0.2.1",
]

IMPORT_CHECKS = {
    "numpy": "numpy",
    "pandas": "pandas",
    "pyarrow": "pyarrow",
    "pulp": "pulp",
    "plotly": "plotly",
    "kaleido": "kaleido",
}

missing_packages = []
for package_name, module_name in IMPORT_CHECKS.items():
    if importlib.util.find_spec(module_name) is None:
        missing_packages.append(package_name)

if missing_packages:
    packages_to_install = [
        package_spec
        for package_spec in PIP_PACKAGES
        if package_spec.split(">=")[0].split("==")[0] in missing_packages
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages_to_install])


In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/content/nhs_rtt_msc_project"),
    Path("/content"),
]
for candidate_root in PROJECT_ROOT_CANDIDATES:
    if (candidate_root / "nhs_rtt_pipeline").exists():
        sys.path.insert(0, str(candidate_root))
        break
else:
    raise FileNotFoundError(
        "Could not find the shared nhs_rtt_pipeline package. "
        "Run this notebook from the project root or upload the full nhs_rtt_msc_project folder to Colab."
    )

from nhs_rtt_pipeline.config import COLUMNS, ensure_directories, get_paths
from nhs_rtt_pipeline.optimisation import (
    DEFAULT_MAX_CAPACITY_INCREASE_PER_TRUST,
    DEFAULT_TOTAL_EXTRA_SESSIONS,
    PATHWAYS_ADDRESSED_PER_THEATRE_SESSION,
    load_optimisation_forecasts,
)

PATHS = get_paths()
ensure_directories(PATHS)

# Step 1 - LP formulation.
# Trust index set: I = Trusts with surgical specialties and available Part 2A decision-to-admit forecasts.
# Decision variables: x_i = integer number of additional treatment sessions allocated to Trust i.
# Objective: minimise sum_i p50 incomplete_decision_to_admit pathways after allocation.
# Constraint (a): sum_i x_i = total_extra_sessions, default 500.
# Constraint (b): x_i >= 0 for every Trust i.
# Constraint (c): x_i <= max_capacity_increase_per_trust, default 50 sessions per Trust.
# Constraint (d): x_i is integer-valued for every Trust i.
# Scenario assumption: pathways_addressed_i <= 8 * x_i and cannot exceed predicted Part 2A demand.
# The full incomplete RTT pathway total is not used as the capacity-simulation objective.

future_optimisation_forecasts = load_optimisation_forecasts(PATHS.future_optimisation_forecasts)
print(f"Loaded Part 2A optimisation forecasts: {PATHS.future_optimisation_forecasts}")
print(f"Rows: {len(future_optimisation_forecasts):,}")
print(f"Trusts: {future_optimisation_forecasts[COLUMNS.trust_code].nunique():,}")
print(
    "Forecast range:",
    future_optimisation_forecasts[COLUMNS.forecast_month].min().date(),
    "to",
    future_optimisation_forecasts[COLUMNS.forecast_month].max().date(),
)
display(future_optimisation_forecasts.head(10))


In [ ]:
from nhs_rtt_pipeline.optimisation import pathway_reduction


example_trust = str(future_optimisation_forecasts[COLUMNS.trust_name].iloc[0])
example_reduction = pathway_reduction(
    trust_name=example_trust,
    extra_sessions=10,
    forecast_df=future_optimisation_forecasts,
    pathways_addressed_per_session=PATHWAYS_ADDRESSED_PER_THEATRE_SESSION,
)
print(example_trust, example_reduction)


In [ ]:
from nhs_rtt_pipeline.optimisation import solve_lp_allocation


allocation_output, allocation_metadata, lp_problem = solve_lp_allocation(
    future_optimisation_forecasts,
    total_extra_sessions=DEFAULT_TOTAL_EXTRA_SESSIONS,
    max_capacity_increase_per_trust=DEFAULT_MAX_CAPACITY_INCREASE_PER_TRUST,
    scenario_column=COLUMNS.p50,
    pathways_addressed_per_session=PATHWAYS_ADDRESSED_PER_THEATRE_SESSION,
)

allocation_output.to_csv(PATHS.lp_allocation_output, index=False)
print(f"Solver status: {allocation_metadata['status']}")
print(f"Saved allocation output to: {PATHS.lp_allocation_output}")
print(allocation_metadata)
display(allocation_output.head(20))


In [ ]:
import plotly.graph_objects as go

from nhs_rtt_pipeline.optimisation import run_sensitivity_analysis


sensitivity_sessions = [100, 250, 500, 750, 1000]
sensitivity_results = run_sensitivity_analysis(
    future_optimisation_forecasts,
    sensitivity_sessions,
    max_capacity_increase_per_trust=DEFAULT_MAX_CAPACITY_INCREASE_PER_TRUST,
    pathways_addressed_per_session=PATHWAYS_ADDRESSED_PER_THEATRE_SESSION,
)
sensitivity_results.to_csv(PATHS.lp_sensitivity_output, index=False)

sensitivity_fig = go.Figure()
sensitivity_fig.add_trace(
    go.Scatter(
        x=sensitivity_results["total_extra_sessions"],
        y=sensitivity_results["total_pathways_addressed"],
        mode="lines+markers",
        line=dict(color="#2563eb", width=3),
        marker=dict(size=8),
        name="Part 2A pathways addressed",
    )
)
sensitivity_fig.update_layout(
    title="Sensitivity of Part 2A Objective to Additional Treatment Sessions",
    xaxis_title="Total additional treatment sessions",
    yaxis_title="Total incomplete decision-to-admit pathways addressed",
    template="plotly_white",
    width=1000,
    height=560,
)
sensitivity_fig.write_image(str(PATHS.lp_sensitivity_png), scale=2)
print(f"Saved sensitivity table to: {PATHS.lp_sensitivity_output}")
print(f"Saved sensitivity chart to: {PATHS.lp_sensitivity_png}")
display(sensitivity_results)


In [ ]:
from nhs_rtt_pipeline.optimisation import run_uncertainty_comparison


uncertainty_comparison = run_uncertainty_comparison(
    future_optimisation_forecasts,
    total_extra_sessions=DEFAULT_TOTAL_EXTRA_SESSIONS,
    max_capacity_increase_per_trust=DEFAULT_MAX_CAPACITY_INCREASE_PER_TRUST,
)
uncertainty_comparison.to_csv(PATHS.lp_uncertainty_comparison, index=False)
print(f"Saved uncertainty comparison to: {PATHS.lp_uncertainty_comparison}")
display(uncertainty_comparison.head(30))


In [ ]:
from nhs_rtt_pipeline.optimisation import run_covid_stress_test


covid_stress_test_results = run_covid_stress_test(
    future_optimisation_forecasts,
    total_extra_sessions=DEFAULT_TOTAL_EXTRA_SESSIONS,
    max_capacity_increase_per_trust=DEFAULT_MAX_CAPACITY_INCREASE_PER_TRUST,
    covid_start="2020-03-01",
    covid_end="2021-09-30",
)
covid_stress_test_results.to_csv(PATHS.lp_covid_stress_test, index=False)
print(f"Saved COVID stress test results to: {PATHS.lp_covid_stress_test}")
display(covid_stress_test_results)
